# floodar — Red Hook, Brooklyn flood exposure

Focused on the **Red Hook peninsula**, clipped to a boundary polygon so areas are Red
Hook *in isolation* — not the bounding box. Uses the whole-city COG at
`../data/nyc_dem_1ft_int_cog.tif`.

DEM: EPSG:2263, whole feet NAVD88, `uint16`. **`0` = water / outside-city / 0-ft ground.**

**Important modeling note:** flooding is computed over the *full* window (so water outside
the polygon can still act as the hydrologic source), and only the **reported area/volume**
is restricted to inside the boundary. Clipping the DEM before flooding would cut off the
water source and under-predict.

In [ ]:
import json
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom
from floodar import io, inspect, viz, flood, query

DEM_PATH = Path('../data/nyc_dem_1ft_int_cog.tif')
assert DEM_PATH.exists(), 'Whole-city COG not found — build it with `floodar cog` (see data/README.md)'

# Active Red Hook boundary (WGS84) -> reproject the polygon to the DEM CRS (2263).
# Swap in an official-tract alternative (see the note below) by changing this filename:
#   redhook_tracts_peninsula.geojson (~591 ac) | redhook_tracts_core.geojson (~546 ac) | redhook_nta_bk33.geojson (~1013 ac)
BOUNDARY = 'redhook_boundary.geojson'   # hand-drawn approximation (~456 ac)
gj = json.load(open(BOUNDARY))
geom = gj['features'][0]['geometry']
geom2263 = transform_geom('EPSG:4326', 'EPSG:2263', geom)
ring = np.array(geom2263['coordinates'][0])

def inside_mask(tr, shape):
    """bool array, True where a cell centre is inside the Red Hook polygon."""
    return ~geometry_mask([geom2263], out_shape=shape, transform=tr)

def draw_boundary(ax, tr, color='red', lw=2):
    cols = (ring[:, 0] - tr.c) / tr.a
    rows = (ring[:, 1] - tr.f) / tr.e
    ax.plot(cols, rows, color=color, lw=lw)

# AOI = polygon bounds + a little padding (so water context is included for connectivity)
minx, maxx = ring[:, 0].min(), ring[:, 0].max()
miny, maxy = ring[:, 1].min(), ring[:, 1].max()
padx, pady = (maxx - minx) * 0.08, (maxy - miny) * 0.08
bounds = (minx - padx, miny - pady, maxx + padx, maxy + pady)

arr, tr = io.read(DEM_PATH, bounds=bounds, max_size=2500)
cell = abs(tr.a)
inside = inside_mask(tr, arr.shape)
print(f'boundary = {BOUNDARY}')
print(f'window {arr.shape}, cell = {cell:.2f} ft, cells inside = {inside.sum():,}')

### Boundary options

The active boundary (`redhook_boundary.geojson`) is a **hand-drawn approximation** of the
peninsula (~456 acres of land). Red Hook has **no standalone official NTA** — NYC's
Neighborhood Tabulation Area for the area is **BK33 'Carroll Gardens-Columbia
Street-Red Hook'** (2010), which also merged into **BK0601** in the 2020 vintage. So the
official geometry that can isolate Red Hook is at the **census-tract** level.

Three official-tract alternatives are saved beside this notebook (dissolved from NYC 2010
census tracts `bmjq-373p` using the tract→NTA equivalency `8ius-dhrr`). To use one, set
`BOUNDARY` above to its filename and re-run:

| file | definition | land area | notes |
|---|---|---|---|
| `redhook_boundary.geojson` *(active)* | hand-drawn peninsula | ~456 ac | editable; not tied to official geometry |
| `redhook_tracts_core.geojson` | tracts **85, 59, 53** | ~546 ac | the three sub-10 ft tracts (Red Hook Houses + waterfront) |
| `redhook_tracts_peninsula.geojson` | tracts **85, 59, 53, 51** | ~591 ac | adds the 15-ft northern neck up to the Gowanus Expressway |
| `redhook_nta_bk33.geojson` | full NTA **BK33** (12 tracts) | ~1013 ac | official label, but includes Carroll Gardens & Columbia St |

Mean tract elevations cleanly separate Red Hook (85≈9, 53≈8, 59≈10, 51≈15 ft) from
Carroll Gardens / Columbia St (25–40 ft). All areas below are recomputed for whichever
boundary is active.

## Land-only stats + histogram (clipped to the boundary, 0 = water masked)

In [ ]:
land = io.read(DEM_PATH, bounds=bounds, max_size=2500, mask_values=0, clip=geom, clip_epsg=4326)[0]
print(inspect.stats(land).as_dict())

counts, edges = inspect.histogram(land, bins=50)
plt.figure(figsize=(8, 3)); plt.bar(edges[:-1], counts, width=np.diff(edges), align='edge')
plt.xlabel('elevation (ft NAVD88)'); plt.ylabel('cells')
plt.title('Red Hook land elevation — most of it sits below ~10 ft');

## Shaded relief with the boundary (water in blue)

In [ ]:
elev = np.ma.filled(arr, 0.0); is_land = elev > 0
base = np.ma.masked_where(~is_land, elev)
hs = viz.hillshade(base, cellsize=cell, z_factor=2.0)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(hs, cmap='gray')
im = ax.imshow(base, cmap='terrain', alpha=0.55, vmax=np.percentile(elev[is_land], 98))
ax.imshow(np.ma.masked_where(is_land, elev), cmap=ListedColormap(['#9ecae1']))
draw_boundary(ax, tr)
ax.set_title('Red Hook — elevation (ft NAVD88), boundary in red'); ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.7, label='ft');

## Flood scenarios (feet NAVD88), area restricted to the boundary

Flooding is solved over the whole window (water outside the polygon is a valid source);
reported acreage counts only cells **inside** the boundary.

In [ ]:
ACRE = 43560.0
levels = [3, 6, 9, 11]
results = flood.scenarios(arr, levels, cell_size=cell, connected=True, baseline=0.0)

fig, axes = plt.subplots(1, len(levels), figsize=(4*len(levels), 4))
for ax, r in zip(axes, results):
    nf = r.newly_flooded & inside                         # dry land flooded, inside boundary
    acres = nf.sum() * cell**2 / ACRE
    print(f"{r.water_level:5.1f} ft  ->  dry-land flooded {acres:7.1f} ac")
    ax.imshow(hs, cmap='gray')
    ax.imshow(np.ma.masked_where(is_land, elev), cmap=ListedColormap(['#c6dbef']))
    depth = np.ma.masked_where(~nf, np.ma.filled(r.depth, 0.0))
    ax.imshow(depth, cmap='Blues', vmin=0, vmax=max(levels))
    draw_boundary(ax, tr, lw=1.5)
    ax.set_title(f'{r.water_level:.0f} ft  ({acres:.0f} ac)'); ax.axis('off')
plt.tight_layout();

## Area-by-elevation table  (clipped — for related-rates & integral work)

Native 1-ft resolution (each pixel = 1 ft²), counted only inside the boundary polygon.
Whole-foot elevations ⇒ 1-ft bands.

| symbol | meaning | calculus role |
|---|---|---|
| `dA/dh` (band area) | land area in band `(h-1, h]` | the **density**; area added per foot |
| `A(h)` (cum. land) | land area with elevation ≤ h `= Σ dA/dh` | the **integral** of the density (hypsometric curve) |
| `A_flood(h)` | *connected* flood footprint on dry land at level h | realistic inundated area |
| `V(h)` | flood-water volume on land at level h | **∫₀ʰ A_flood(z) dz**, so `dV/dh = A_flood(h)` |

* **Related rates** — water rising at `dh/dt` gives `dA/dt = (dA/dh)·(dh/dt)` and `dV/dt = A_flood(h)·(dh/dt)`.
* **Integration** — `A(h)` is the running integral of `dA/dh`; `V(h)` is the running integral of `A_flood`.

Saved to `redhook_area_by_elevation.csv`.

In [ ]:
import csv
from IPython.display import HTML, display

# native-resolution read of the AOI (each pixel = 1 ft^2); flood still sees all water
arr_hi, tr_hi = io.read(DEM_PATH, bounds=bounds, max_size=100000)
cell_hi = abs(tr_hi.a); cell_area = cell_hi**2
inside_hi = inside_mask(tr_hi, arr_hi.shape)
elev_hi = np.ma.filled(arr_hi, 0).astype(int)
land_hi = (elev_hi > 0) & inside_hi                     # dry land inside boundary
total_land_acres = land_hi.sum() * cell_area / ACRE

MAXH = 16
band_counts = np.bincount(elev_hi[land_hi], minlength=MAXH + 2)
band_area = band_counts * cell_area / ACRE               # acres per 1-ft band (dA/dh)
cum_area = np.cumsum(band_area)                           # A(h) = land area <= h

rows, prev = [], 0.0
for h in range(0, MAXH + 1):
    r = flood.bathtub(arr_hi, float(h), cell_size=cell_hi, baseline=0.0, connected=True)
    nf = r.newly_flooded & inside_hi                      # flooded dry land inside boundary
    A_flood = nf.sum() * cell_area / ACRE
    V = (np.ma.filled(r.depth, 0.0) * nf).sum() * cell_area / ACRE   # acre-feet
    rows.append((h, band_area[h], cum_area[h], A_flood, A_flood - prev, V))
    prev = A_flood

csv_path = Path('redhook_area_by_elevation.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['elev_ft_navd88', 'band_area_acres_per_ft', 'cum_land_area_acres',
                'connected_flood_area_acres', 'marginal_flood_area_acres_per_ft',
                'flood_volume_acre_ft'])
    for h, ba, ca, fa, dfa, v in rows:
        w.writerow([h, f'{ba:.4f}', f'{ca:.4f}', f'{fa:.4f}', f'{dfa:.4f}', f'{v:.4f}'])

print(f'dry-land area (all elevations, inside boundary): {total_land_acres:,.1f} acres')
print(f'saved -> {csv_path.resolve()}')

hdr = ['h (ft)', 'band dA/dh<br>(ac/ft)', 'cum A(h)<br>(ac)', 'flood A(h)<br>(ac)',
       '\u0394flood<br>(ac/ft)', 'volume V(h)<br>(ac\u00b7ft)']
html = ['<table><tr>' + ''.join(f'<th>{c}</th>' for c in hdr) + '</tr>']
for h, ba, ca, fa, dfa, v in rows:
    html.append(f'<tr><td align=right>{h}</td><td align=right>{ba:.2f}</td>'
                f'<td align=right>{ca:.2f}</td><td align=right>{fa:.2f}</td>'
                f'<td align=right>{dfa:.2f}</td><td align=right>{v:.2f}</td></tr>')
html.append('</table>')
display(HTML(''.join(html)))

In [ ]:
hs_ = [r[0] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hs_, [r[3] for r in rows], 'o-', label='connected flood area  A_flood(h)')
ax[0].plot(hs_, [r[2] for r in rows], 's--', color='gray', label='land area \u2264 h  (hypsometric)')
ax[0].set_xlabel('water level  h  (ft NAVD88)'); ax[0].set_ylabel('area (acres)')
ax[0].set_title('Inundated area vs level  \u2014  V(h)=\u222bA dz'); ax[0].legend()
ax[1].bar(hs_, [r[4] for r in rows], width=0.8)
ax[1].set_xlabel('water level  h  (ft NAVD88)'); ax[1].set_ylabel('\u0394A per foot (acres/ft)')
ax[1].set_title('Marginal flood area  dA/dh  \u2014  the related-rates density')
plt.tight_layout();

## Elevation profile — a cross-section through the peninsula

In [ ]:
start = (bounds[0] + 5, (bounds[1] + bounds[3]) / 2)
end   = (bounds[2] - 5, (bounds[1] + bounds[3]) / 2)
dist, elevs = query.profile(DEM_PATH, start, end, n=400, src_epsg=2263)
plt.figure(figsize=(9, 3)); plt.plot(dist, elevs)
plt.axhline(11, color='C0', ls='--', label='11 ft (Sandy surge)'); plt.legend()
plt.xlabel('distance (ft)'); plt.ylabel('elevation (ft NAVD88)')
plt.title('Red Hook W-E cross-section');